# TNMRControl Basic Usage

Simple notebook for opening a vetted canonical `.tnt` file, inspecting it, running acquisition, and stopping acquisition.

## 1. Setup

In [ ]:
from pathlib import Path
import ctypes
import importlib
import sys

for repo_root in (Path.cwd(), *Path.cwd().parents):
    redstone_dir = repo_root / "hardware_capture" / "rf" / "redstone"
    if redstone_dir.is_dir():
        break
else:
    raise FileNotFoundError("Run this notebook from the Hardware Capture repository.")

if str(redstone_dir) not in sys.path:
    sys.path.insert(0, str(redstone_dir))

import tnmrcontrol
importlib.reload(tnmrcontrol)

from tnmrcontrol import TNMRControl

print("Python running as admin:", bool(ctypes.windll.shell32.IsUserAnAdmin()))

## 2. Select TNT File

In [ ]:
input_tnt_text = r""  # Set the vetted canonical TNT path explicitly.

if not input_tnt_text.strip():
    raise RuntimeError("Set input_tnt_text to a vetted canonical .tnt before continuing.")

input_tnt = Path(input_tnt_text).expanduser().resolve()
assert input_tnt.exists(), input_tnt
assert input_tnt.suffix.lower() == ".tnt", input_tnt
input_tnt

## 3. Connect to TNMR Application

In [ ]:
# connect(): attach to an existing TNMR app; it does not start TNMR.
# This does not open a TNT file.
control = TNMRControl()
control.connect()
print("TNMR application connected")

## 4. Check TNMR Application Access

In [ ]:
info = control.application_info()

for key, value in info.items():
    print(f"{key}: {value}")

## 5. Open TNT File

In [ ]:
control.open(input_tnt)

print("Connected TNT:", control.path)
print("Embedded sequence (informational):", control.sequence_name())

## 6. Check Session and Parameters

In [ ]:
print("Active path:", control.active_document_path())

try:
    print("Open documents:")
    for item in control.document_list():
        print(" -", item)
except RuntimeError as exc:
    print("Document list unavailable:", exc)

for name in ("Observe Freq.", "Nucleus", "Scans 1D", "Points 1D"):
    try:
        print(f"{name}: {control.get_parameter(name)}")
    except RuntimeError as exc:
        print(f"{name}: unavailable ({exc})")

## 7. Optional Zero and Go

In [ ]:
# Run only after verifying the opened TNMR file and instrument state.
# control.zero_and_go()
# print("ZG command sent.")

## 8. Optional Stop or Abort During Acquisition

In [ ]:
# Use this only while an acquisition is still running.
# The TNMR manual lists Stop as an Application/Document command;
# the wrapper sends it through the TNMR app when connected.
# control.stop()
# print("Stop command sent.")

# Hard abort only if needed:
# control.abort()

## 9. Optional Save

In [ ]:
# Save in place:
# control.save()

# Save to a new TNT file:
# output_tnt = input_tnt.with_name(input_tnt.stem + "_copy.tnt")
# control.save_as(output_tnt)
# print("Saved:", output_tnt)